# S03 - Sahm-rule unemployment signal (macro)

**Hypothesis.** A recession is under way once the 3-month average unemployment rate rises 0.5 pp
above its low of the previous 12 months (Sahm 2019). Recessions coincide with most deep bear
markets, so holding T-bills while the rule is triggered should cut drawdowns.

| Variant | Risk-on (hold S&P 500) when ... |
|---|---|
| comp_thr05 / comp_thr03 | Sahm indicator from current-vintage UNRATE < 0.5 / 0.3 |
| rt_thr05 | FRED real-time Sahm indicator (built from first-release data) < 0.5 |
| rt_hyst | real-time: risk-off when >= 0.5, back on only once < 0.25 |

**Inputs and point-in-time lags**

| Series | Source | Start | Lag used |
|---|---|---|---|
| Unemployment rate, monthly (current vintage, revised) | FRED `UNRATE` | 1948 | month M usable from M+1 + 7 days |
| Real-time Sahm indicator (first-release data) | FRED `SAHMREALTIME` | 1959-12 | month M usable from M+1 + 7 days |

The employment report comes out on the first Friday of M+1, so day 7 is conservative.
The current-vintage variants carry a small revision look-ahead (seasonal revisions, a few
tenths at most); the real-time variants do not. Monthly check, next-close, 5 bp per switch.
Before the data start the strategy has no signal, so its periods begin in 1949 / 1960 and
the benchmarks are compared on the same rows.

**Sources.** Sahm (2019), *Direct stimulus payments to individuals*, Hamilton Project /
Brookings; FRED series `SAHMREALTIME` and `UNRATE`, https://fred.stlouisfed.org.

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S03', 'sahm_rule', 'macro/sahm_rule.ipynb'
PROTOCOL, REBALANCE = 'main', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 9


## 1. Data and point-in-time inputs

In [2]:
unrate = bt_long.fred_series("UNRATE")
sahm_rt = bt_long.fred_series("SAHMREALTIME")
inputs = {"px": px, "unrate": unrate, "sahm_rt": sahm_rt}
print("UNRATE", unrate.index[0].date(), "->", unrate.index[-1].date(), "| SAHMREALTIME", sahm_rt.index[0].date(), "->", sahm_rt.index[-1].date())

UNRATE 1948-01-01 -> 2026-09-01 | SAHMREALTIME 1959-12-01 -> 2026-09-01


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
LAG = dict(months=0, days=7)


def sahm_from_unrate(u):
    u3 = u.rolling(3).mean()
    return u3 - u3.shift(1).rolling(12).min()


def hysteresis(ind, off=0.5, on=0.25):
    out, state = [], 1.0
    for v in ind.to_numpy():
        if np.isnan(v):
            out.append(np.nan)
            continue
        state = 0.0 if v >= off else (1.0 if v < on else state)
        out.append(state)
    return pd.Series(out, index=ind.index)


def build_signal(inputs, source="comp", thr=0.5, hyst=False):
    ind = sahm_from_unrate(inputs["unrate"]) if source == "comp" else inputs["sahm_rt"]
    ind = ind.dropna()
    m = hysteresis(ind, thr, thr / 2) if hyst else (ind < thr).astype(float)
    return bt_pit.monthly_to_daily_pit(m, inputs["px"].index, **LAG)


VARIANTS = {
    "comp_thr05": dict(source="comp", thr=0.5),
    "comp_thr03": dict(source="comp", thr=0.3),
    "rt_thr05": dict(source="rt", thr=0.5),
    "rt_hyst": dict(source="rt", thr=0.5, hyst=True),
}
PARAMS_TEXT = {n: f"{n}; FRED UNRATE/SAHMREALTIME, lag M+1+7d; monthly; next_close; 5bp" for n in VARIANTS}

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=sahm_rt.index, y=sahm_rt, name="real-time Sahm"))
comp = sahm_from_unrate(unrate)
fig.add_trace(go.Scatter(x=comp.index, y=comp, name="Sahm from current UNRATE", opacity=0.6))
fig.add_hline(y=0.5, line_dash="dash", line_color="red")
fig.update_layout(title="Sahm recession indicator (pp)", height=380, legend=dict(orientation="h"))
fig.show()

## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [5]:
N_CUTS = 30
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
comp_thr05,30,0,0.0,True
comp_thr03,30,0,0.0,True
rt_thr05,30,0,0.0,True
rt_hyst,30,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [6]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
comp_thr05,0.207449,-0.496294,-0.34121,0.741935,0.428224,1950-04-18
comp_thr03,0.336761,-0.366982,-0.211898,0.68884,0.579362,1950-04-18
rt_thr05,0.130927,-0.374071,-0.205956,0.742896,0.517889,1961-01-11
rt_hyst,0.219106,-0.285893,-0.117777,0.711187,0.379785,1961-01-11


chosen on dev Sharpe only: comp_thr03 | comp_thr03; FRED UNRATE/SAHMREALTIME, lag M+1+7d; monthly; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [7]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.337,0.549,-0.212,0.704,-0.367,0.557,-0.220
val,0.457,0.319,0.138,0.572,-0.115,0.309,0.148
holdout,0.688,0.781,-0.092,0.658,0.031,0.779,-0.091
full,0.461,0.527,-0.066,0.654,-0.193,0.528,-0.066


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.085066,0.108082,0.336761,0.457012,-0.329581,0.258103,-0.204547,1987-08-25,1989-05-17,25.560265,0.68884,23,0.579362,0.00029
val,0.090155,0.125415,0.45732,0.650709,-0.269632,0.334363,-0.068651,2000-09-01,2003-05-30,5.617723,0.675392,7,0.350096,0.000175
holdout,0.115908,0.156193,0.688373,0.973852,-0.337905,0.34302,-0.119799,2020-02-19,2021-10-29,6.30391,0.812027,11,0.65521,0.000328
full,0.093071,0.124623,0.461275,0.642754,-0.337905,0.275435,-0.204547,2020-02-19,2021-10-29,905.181463,0.712349,41,0.535921,0.000268


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.337,-0.367,-0.212,0.994
val,0.457,-0.115,0.138,0.762
holdout,0.688,0.031,-0.092,NaN
full,0.461,-0.193,-0.066,NaN


n_trials=13 | DSR=0.988 | passes=False | holdout confirms=True


dev_beats_sma200        False
dev_beats_bh            False
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh             True
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos     True
Name: check, dtype: bool

In [8]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [9]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.460,0.092,-0.338,0.53
5,0.458,0.092,-0.338,0.53
10,0.456,0.092,-0.338,0.53
25,0.450,0.091,-0.338,0.53
50,0.439,0.089,-0.338,0.53


break-even one-way cost vs SMA200 Sharpe over the same window (0.654): 0.0 bp


,placebo
stat,Sharpe
real,0.458283
placebo_mean,0.387984
placebo_q95,0.510826
quantile,0.795
p_one_sided,0.205
n_placebo,200
time_in_market,0.589785


peek test: Sharpe by shift {-21: 0.432, -5: 0.46, -1: 0.458, 0: 0.458, 1: 0.458, 5: 0.458, 21: 0.405} | gain -0.026


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1950s,0.87,1.39,1.47,0.10,0.19,0.18,0.65
1960s,0.23,0.40,0.43,0.06,0.08,0.07,0.81
1970s,-0.21,0.03,0.37,0.04,0.06,0.09,0.61
1980s,0.45,0.53,0.58,0.15,0.17,0.17,0.68
1990s,0.73,0.92,0.64,0.14,0.18,0.13,0.78
2000s,0.17,-0.05,0.50,0.04,-0.01,0.08,0.57
2010s,0.97,0.90,0.71,0.14,0.14,0.09,0.93
2020s,0.39,0.67,0.58,0.08,0.15,0.10,0.63


## 7. Register the result

In [10]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
35,S03-v1/dev,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"comp_thr05; FRED UNRATE/SAHMREALTIME, lag M+1+...",dev,0.2074,-0.4963,-0.3412,,,,,run,"dev-only sweep, variant comp_thr05"
36,S03-v2/dev,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"comp_thr03; FRED UNRATE/SAHMREALTIME, lag M+1+...",dev,0.3368,-0.3670,-0.2119,,,,,run,"dev-only sweep, variant comp_thr03"
37,S03-v3/dev,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"rt_thr05; FRED UNRATE/SAHMREALTIME, lag M+1+7d...",dev,0.1309,-0.3741,-0.2060,,,,,run,"dev-only sweep, variant rt_thr05"
38,S03-v4/dev,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"rt_hyst; FRED UNRATE/SAHMREALTIME, lag M+1+7d;...",dev,0.2191,-0.2859,-0.1178,,,,,run,"dev-only sweep, variant rt_hyst"
39,S03/dev,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"comp_thr03; FRED UNRATE/SAHMREALTIME, lag M+1+...",dev,0.3368,-0.3670,-0.2119,0.9944,,13,False,fail,fails frozen metric; chosen comp_thr03 on dev;...
40,S03/val,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"comp_thr03; FRED UNRATE/SAHMREALTIME, lag M+1+...",val,0.4573,-0.1149,0.1383,0.7619,,13,False,fail,fails frozen metric; chosen comp_thr03 on dev;...
41,S03/holdout,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"comp_thr03; FRED UNRATE/SAHMREALTIME, lag M+1+...",holdout,0.6884,0.0308,-0.0924,,,13,False,fail,fails frozen metric; chosen comp_thr03 on dev;...
42,S03/full,2026-10-20,sahm_rule,macro/sahm_rule.ipynb,"comp_thr03; FRED UNRATE/SAHMREALTIME, lag M+1+...",full,0.4613,-0.1929,-0.0658,,0.9875,13,False,fail,fails frozen metric; chosen comp_thr03 on dev;...


registry trial count: 13


## 8. Current signal

In [11]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (comp_thr03) on 2026-10-19: equity weight 1.00


,equity weight at decision
Date,
2026-04-30,1.0
2026-05-29,1.0
2026-06-30,1.0
2026-07-31,1.0
2026-08-31,1.0
2026-09-30,1.0


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [12]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 30 cut dates; peek test gain -0.026.
* dev: Sharpe 0.337 (vs SMA200 -0.367, vs B&H -0.212, p vs SMA200 0.994)
* val: Sharpe 0.457 (vs SMA200 -0.115, vs B&H +0.138, p vs SMA200 0.762)
* holdout: Sharpe 0.688 (vs SMA200 +0.031, vs B&H -0.092)
* Deflated Sharpe 0.988 with n_trials=13.
* Placebo: real 0.458 vs placebo 95th pct 0.511 (p 0.205); break-even cost vs SMA200 full 0.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_beats_sma200, dev_beats_bh, dev_p_le_0.1, val_beats_sma200, val_p_le_0.1); holdout confirms.


## 10. Caveats

* The rule is a coincident recession signal: it usually triggers months after the equity peak, and
  equities often bottom before unemployment peaks, so it can sell late and buy back late.
* Few events (about 11 recessions since 1948), so any Sharpe difference has a wide confidence interval.
* Current-vintage UNRATE is revised; the real-time variants avoid that look-ahead.
* Data start 1948 / 1959: the dev period is shorter than for price-based rules.

## 11. Interpretation

* Every variant loses to **buy & hold** over its own dev rows (Sharpe difference -0.12 to -0.34), and the
  chosen `comp_thr03` (dev from 1950) trails SMA200 by 0.37 in dev (bootstrap p 0.99). **Fails** clearly.
* The Sahm rule is a *coincident* recession signal: by the time unemployment has risen 0.3-0.5 pp the
  market has usually fallen, and it stays risk-off into the recovery, when equities rebound before
  unemployment peaks. It sells late and buys back late, which is the worst combination for timing.
* The real-time variants (FRED `SAHMREALTIME`, first-release data, dev from 1961) also lose to B&H over
  their window (-0.21 and -0.12), so data revisions are not what makes the rule look bad.
* Placebo p 0.21: the timing is not reliably better than random regimes with the same time in market.